# System doboru modeli predykcyjnych szeregów czasowych (XGBoost + Meta-Learning)
**Cel:** Narzędzie do optymalizacji predykcji finansowych przy użyciu metod klasycznych i ML.

#### Zadanie 1: Wybór 6 modeli predykcyjnych szeregów czasowych
 W ramach projektu zidentyfikowaliśmy 6 kluczowych architektur, które zostaną zaimplementowane i porównane. Każdy model posiada specyficzne hiperparametry

#### Zadanie 2: Cel predykcji Dla chwili czasowej **t+1** przewidujemy:

1. **Główny cel:** Dokładną wartość zamknięcia (`Close`) – podejście regresyjne.
2. **Cel pomocniczy:** Kierunek zmiany (Wzrost/Spadek) w celu filtrowania transakcji przez Meta-Model.

---

## 1. Importy i Konfiguracja

In [ ]:
import pandas as pd
import pandas_ta as ta
import yfinance as yf
import numpy as np
import matplotlib.pyplot as plt
import shap
import joblib
import os

from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import mean_absolute_error, classification_report, precision_score

plt.style.use('seaborn-v0_8')
os.makedirs('stages', exist_ok=True)

## 2: Przygotowanie Danych 
(Feature Engineering)Pobieramy dane dla **Apple (AAPL)** oraz indeksu strachu **VIX** jako kontekstu rynkowego. Tworzymy cechy techniczne (RSI, SMA, ATR) oraz opóźnienia (Lags), które są niezbędne dla modeli nie-rekurencyjnych jak XGBoost.

### Pobieranie i Przetwarzanie Danych:

In [ ]:
print("Pobieranie danych...")
tickers = ['AAPL', '^VIX']
data = yf.download(tickers, start='2000-01-01', group_by='ticker', auto_adjust=False)

df = data['AAPL'].copy()
vix = data['^VIX']['Close'].copy()

df['RSI'] = df.ta.rsi(length=14)
df['SMA_20'] = df.ta.sma(length=20)
df['SMA_50'] = df.ta.sma(length=50)
df['ATR'] = df.ta.atr(length=14)
df['Dist_SMA'] = (df['Close'] - df['SMA_50']) / df['SMA_50']

df['VIX'] = vix
df['VIX'] = df['VIX'].ffill()
df['Panic_Mode'] = (df['VIX'] > 25).astype(int)
df['Rolling_Std'] = df['Close'].rolling(window=20).std()
df['VIX_Slope'] = df['VIX'].diff(5)

bb = df.ta.bbands(length=20, std=2)
df['BB_Width'] = bb['BBB_20_2.0_2.0']
df['RSI_Dist'] = abs(df['RSI'] - 50)

for lag in [1, 2, 3, 5]:
    df[f'Close_Lag_{lag}'] = df['Close'].shift(lag)

df['Target'] = df['Close'].shift(-1)
df.dropna(inplace=True)

print(f"Dane gotowe. Liczba wierszy: {len(df)}")
df.tail()

Base Model Accuracy: 44.73%


## 2. Feature Engineering: Market Regime
The core hypothesis is that the Base Model fails during specific market regimes (e.g., high volatility).
We introduce "Meta-Features" to detect these regimes:
- **VIX_Slope**: Is fear rising?
- **BB_Width**: Is the market volatile?
- **RSI_Dist**: Is the trend overextended?
- **Model_Confidence**: How confident is the Base Model?


In [11]:
# Calculate Meta-Features (already in CSV from step4, but let's visualize)
meta_features = ['RSI', 'ATR', 'Rolling_Std', 'Model_Confidence', 'VIX_Slope', 'BB_Width', 'RSI_Dist']

# Correlation Matrix
plt.figure(figsize=(10, 8))
sns.heatmap(df[meta_features].corr(), annot=True, cmap='coolwarm')
plt.title('Correlation of Meta-Features')
plt.show()


KeyError: "['Rolling_Std', 'Model_Confidence', 'VIX_Slope', 'BB_Width', 'RSI_Dist'] not in index"

<Figure size 1000x800 with 0 Axes>

## 3. Meta-Model Training
We train a Random Forest Classifier to predict `Meta_Target` using the Meta-Features.
We use `GridSearchCV` to optimize for Precision (we want to be sure when we trade).


In [ ]:
X_meta = df[meta_features]
y_meta = df['Meta_Target']

# Split Data (50/50)
split = int(len(df) * 0.5)
X_train, X_test = X_meta.iloc[:split], X_meta.iloc[split:]
y_train, y_test = y_meta.iloc[:split], y_meta.iloc[split:]

# Train Optimized Model (Parameters from Step 5)
rf = RandomForestClassifier(n_estimators=100, max_depth=4, min_samples_leaf=3, class_weight='balanced', random_state=42)
rf.fit(X_train, y_train)

# Predict
meta_preds = rf.predict(X_test)

print("Meta-Model Performance:")
print(classification_report(y_test, meta_preds))


## 4. Results & Equity Curve
We simulate the trading strategy:
- **Base Strategy**: Trade every signal from the Base Model.
- **Meta Strategy**: Trade only when the Meta-Model predicts "1" (High Probability of Success).


In [ ]:
# Simulation
test_data = df.iloc[split:].copy()
test_data['Meta_Filter'] = meta_preds

# Calculate Returns
test_data['Return'] = (test_data['Actual_Close'] - test_data['Prev_Close']) / test_data['Prev_Close']
test_data['Strategy_Base'] = test_data['Return'] * test_data['Signal']
test_data['Strategy_Meta'] = test_data['Strategy_Base'] * test_data['Meta_Filter']

# Cumulative Returns
test_data['Equity_Base'] = (1 + test_data['Strategy_Base']).cumprod()
test_data['Equity_Meta'] = (1 + test_data['Strategy_Meta']).cumprod()

# Plot
plt.figure(figsize=(12, 6))
plt.plot(test_data['Equity_Base'], label='Base Strategy (Regression Only)', color='red', alpha=0.6)
plt.plot(test_data['Equity_Meta'], label='Meta-Labeling Strategy (Hybrid)', color='green', linewidth=2)
plt.title('Equity Curve Comparison')
plt.legend()
plt.show()


## 5. Explainability: Why it works?
Which features are most important for the filter?


In [ ]:
importances = pd.Series(rf.feature_importances_, index=meta_features)
importances.sort_values().plot(kind='barh', color='teal')
plt.title('Feature Importance for Meta-Model')
plt.show()
